In [1]:
import pandas as pd, numpy as np

HORIZON = 7   # forecast horizon (days). Lags aur rolling features is se kam nahi honge

df = pd.read_parquet("../data/processed/cleaned_merged.parquet")
holidays = pd.read_csv("../data/raw/holidays_events.csv", parse_dates=["date"])
n0 = len(df)

# ---- 1. Calendar table (train + test period ki saari dates) ----
cal = pd.DataFrame({"date": pd.date_range("2013-01-01", "2017-08-31")})
d = cal["date"]

cal["day_of_week"]  = d.dt.dayofweek
cal["day_of_month"] = d.dt.day
cal["week_of_year"] = d.dt.isocalendar().week.astype(int)
cal["month"]        = d.dt.month
cal["quarter"]      = d.dt.quarter
cal["year"]         = d.dt.year
cal["is_weekend"]   = (cal["day_of_week"] >= 5).astype(int)
cal["is_payday"]    = ((d.dt.day == 15) | d.dt.is_month_end).astype(int)
cal["is_new_year"]  = ((d.dt.month == 1) & (d.dt.day == 1)).astype(int)

# agle 25 December tak din (25 Dec ke baad agle saal ka)
xmas = pd.to_datetime(d.dt.year.astype(str) + "-12-25")
xmas_next = pd.to_datetime((d.dt.year + 1).astype(str) + "-12-25")
cal["days_to_christmas"] = (xmas.where(d <= xmas, xmas_next) - d).dt.days

# EDA se: earthquake window aur Mother's Day (May ka 2nd Sunday)
cal["is_earthquake"]  = d.between("2016-04-16", "2016-05-16").astype(int)
cal["is_mothers_day"] = ((d.dt.month == 5) & (d.dt.dayofweek == 6) & d.dt.day.between(8, 14)).astype(int)

# ---- 2. Holiday proximity (sirf National, asli chhutti wale types) ----
HOL_TYPES = ["Holiday", "Transfer", "Additional", "Bridge"]
hol = holidays[(holidays["transferred"] == False)
               & holidays["type"].isin(HOL_TYPES)
               & (holidays["locale"] == "National")]
hd = hol["date"].drop_duplicates().sort_values().to_numpy().astype("datetime64[ns]")
dv = cal["date"].to_numpy().astype("datetime64[ns]")

nxt = hd[np.clip(np.searchsorted(hd, dv, side="left"), 0, len(hd) - 1)]
prv = hd[np.clip(np.searchsorted(hd, dv, side="right") - 1, 0, len(hd) - 1)]
cal["days_to_next_holiday"]   = ((nxt - dv) / np.timedelta64(1, "D")).astype(int).clip(0, 30)
cal["days_since_prev_holiday"] = ((dv - prv) / np.timedelta64(1, "D")).astype(int).clip(0, 30)

# ---- 3. Chhote dtypes + merge ----
for c in cal.columns[1:]:
    cal[c] = cal[c].astype("int16")

df = df.merge(cal, on="date", how="left")
assert len(df) == n0, "merge se rows badh gayi!"

# ---- 4. Checks ----
new_cols = list(cal.columns[1:])
print("shape:", df.shape, "| NaN in new cols:", int(df[new_cols].isna().sum().sum()))
print("\nFlag counts (days):")
print(cal[["is_weekend", "is_payday", "is_new_year", "is_earthquake", "is_mothers_day"]].sum().to_string())
print("\nSample around Christmas / New Year 2016:")
print(cal[cal["date"].between("2016-12-22", "2017-01-02")]
      [["date", "day_of_week", "days_to_christmas", "days_to_next_holiday", "days_since_prev_holiday", "is_new_year"]]
      .to_string(index=False))

shape: (3000888, 32) | NaN in new cols: 0

Flag counts (days):
is_weekend        486
is_payday         112
is_new_year         5
is_earthquake      31
is_mothers_day      5

Sample around Christmas / New Year 2016:
      date  day_of_week  days_to_christmas  days_to_next_holiday  days_since_prev_holiday  is_new_year
2016-12-22            3                  3                     0                        0            0
2016-12-23            4                  2                     0                        0            0
2016-12-24            5                  1                     0                        0            0
2016-12-25            6                  0                     0                        0            0
2016-12-26            0                364                     0                        0            0
2016-12-27            1                363                     4                        1            0
2016-12-28            2                362                     3

In [2]:
# ---- 1. Wide table: rows = har calendar din (25 Dec bhi, NaN), columns = (store, family) ----
START    = pd.Timestamp("2013-01-01")
full_dates = pd.date_range(START, df["date"].max())

wide = (df.set_index(["date", "store_nbr", "family"])["sales"]
          .unstack(["store_nbr", "family"])
          .reindex(full_dates))
wide.columns = pd.MultiIndex.from_arrays([
    wide.columns.get_level_values(0).astype(int),
    wide.columns.get_level_values(1).astype(str),
])
print("wide shape:", wide.shape, "| NaN days (25 Dec):", int(wide.isna().all(axis=1).sum()))

# ---- 2. Har df row ko wide table mein uski jagah se jodo (merge nahi, fast) ----
keys    = pd.MultiIndex.from_arrays([df["store_nbr"].astype(int).to_numpy(),
                                     df["family"].astype(str).to_numpy()])
col_idx = wide.columns.get_indexer(keys)
row_idx = (df["date"] - START).dt.days.to_numpy()
assert (col_idx >= 0).all(), "kuch store-family match nahi hue"

def take(w):
    return w.to_numpy(dtype="float32")[row_idx, col_idx]

# ---- 3. Sales lags (sab >= HORIZON) ----
LAGS = [7, 14, 21, 28]
assert min(LAGS) >= HORIZON
for lag in LAGS:
    df[f"sales_lag_{lag}d"] = take(wide.shift(lag))

# same weekday ke pichle 4 hafton ka average
df["sales_same_dow_mean_4w"] = df[[f"sales_lag_{l}d" for l in LAGS]].mean(axis=1)

# ---- 4. Rolling mean/std: window HORIZON din shift karke (taake 7 din purane data par khatam ho) ----
base = wide.shift(HORIZON)
for win in [7, 14, 28]:
    r = base.rolling(win, min_periods=int(win * 0.75))   # 25 Dec ke NaN se kharab na ho
    df[f"rolling_mean_{win}d"] = take(r.mean())
    df[f"rolling_std_{win}d"]  = take(r.std())

# ---- 5. Checks ----
new = [c for c in df.columns if c.startswith(("sales_lag_", "sales_same_dow", "rolling_"))]
print("\nNaN per new feature:")
print(df[new].isna().sum().to_string())

# lag sahi hai? df se seedha 7 din purani sales dekh ke compare
chk = df.sample(5000, random_state=1)[["date", "store_nbr", "family", "sales_lag_7d"]].copy()
chk["date7"] = chk["date"] - pd.Timedelta(days=7)
ref = df[["date", "store_nbr", "family", "sales"]].rename(columns={"date": "date7", "sales": "ref"})
chk = chk.merge(ref, on=["date7", "store_nbr", "family"], how="left")
ok = chk.dropna(subset=["sales_lag_7d", "ref"])
print("\nlag_7d matches true sales 7 days earlier:", bool(np.allclose(ok["sales_lag_7d"], ok["ref"], atol=1e-2)), "| rows checked:", len(ok))

# 25 Dec ke aas paas: 1 Jan 2014 ka lag_7 NaN hona chahiye (7 din pehle 25 Dec tha), 2 Jan ka 26 Dec se
g = df[(df["store_nbr"] == 1) & (df["family"] == "GROCERY I") & df["date"].between("2013-12-24", "2014-01-03")]
print("\nStore 1, GROCERY I around 25 Dec 2013:")
print(g[["date", "sales", "sales_lag_7d"]].to_string(index=False))

wide shape: (1688, 1782) | NaN days (25 Dec): 4

NaN per new feature:
sales_lag_7d              19602
sales_lag_14d             32076
sales_lag_21d             44550
sales_lag_28d             57024
sales_same_dow_mean_4w    12474
rolling_mean_7d           19602
rolling_std_7d            19602
rolling_mean_14d          28512
rolling_std_14d           28512
rolling_mean_28d          48114
rolling_std_28d           48114

lag_7d matches true sales 7 days earlier: True | rows checked: 4962

Store 1, GROCERY I around 25 Dec 2013:
      date  sales  sales_lag_7d
2013-12-24 3036.0        2713.0
2013-12-26 2052.0        3030.0
2013-12-27 2152.0        3972.0
2013-12-28 1323.0        2916.0
2013-12-29  529.0        2136.0
2013-12-30 1485.0        4367.0
2013-12-31 1490.0        3036.0
2014-01-01    0.0           NaN
2014-01-02 2441.0        2052.0
2014-01-03 2117.0        2152.0


In [3]:
# ---- 1. Promo ka wide table (Cell 2 jaisa hi layout) ----
wide_p = (df.set_index(["date", "store_nbr", "family"])["onpromotion"]
            .unstack(["store_nbr", "family"]).reindex(full_dates))
wide_p.columns = pd.MultiIndex.from_arrays([
    wide_p.columns.get_level_values(0).astype(int),
    wide_p.columns.get_level_values(1).astype(str),
])
wide_p = wide_p.reindex(columns=wide.columns).fillna(0)      # 25 Dec = 0 promo
assert list(wide_p.columns) == list(wide.columns)

def take_p(arr):
    return np.asarray(arr, dtype="float32")[row_idx, col_idx]

P  = wide_p.to_numpy() > 0
Pf = wide_p.gt(0).astype("float32")

# ---- 2. Basic promo features ----
df["promo_flag"]         = (df["onpromotion"] > 0).astype("int8")
df["log_onpromotion"]    = np.log1p(df["onpromotion"]).astype("float32")
df["promo_recorded"]     = (df["date"] >= "2014-04-01").astype("int8")   # EDA: pehle promo record hi nahi hota tha
df["onpromotion_lag_7d"] = take_p(wide_p.shift(HORIZON).to_numpy())

# ---- 3. Duration: abhi tak lagatar kitne din se promo par (aaj shamil) ----
streak = np.zeros(P.shape, dtype="int16")
streak[0] = P[0]
for t in range(1, P.shape[0]):
    streak[t] = np.where(P[t], streak[t - 1] + 1, 0)
df["promo_streak"] = streak[row_idx, col_idx]

# ---- 4. Frequency: pichle 7 / 28 din mein kitne din promo par (aaj shamil) ----
df["promo_days_7d"]  = take_p(Pf.rolling(7,  min_periods=1).sum().to_numpy())
df["promo_days_28d"] = take_p(Pf.rolling(28, min_periods=1).sum().to_numpy())

# ---- 5. Intensity: aaj ka promo count apni pichli 28 din ki average ke muqable ----
base28 = wide_p.shift(1).rolling(28, min_periods=7).mean()
df["promo_intensity_vs_28d"] = take_p((wide_p / (base28 + 1)).to_numpy())

# ---- 6. Breadth: us din store ki kitni families / family ke kitne stores promo par ----
df["_p"] = df["promo_flag"].astype("int16")
df["store_promo_families"] = df.groupby(["store_nbr", "date"], observed=True)["_p"].transform("sum").astype("int16")
df["family_promo_stores"]  = df.groupby(["family", "date"],    observed=True)["_p"].transform("sum").astype("int16")
df = df.drop(columns="_p")

# ---- 7. Checks ----
promo_cols = ["promo_flag", "log_onpromotion", "promo_recorded", "onpromotion_lag_7d", "promo_streak",
              "promo_days_7d", "promo_days_28d", "promo_intensity_vs_28d",
              "store_promo_families", "family_promo_stores"]
print("shape:", df.shape)
print("\nNaN per promo feature:"); print(df[promo_cols].isna().sum().to_string())
print("\nstreak == 0 exactly when no promo:", bool(((df["promo_streak"] == 0) == (df["onpromotion"] == 0)).all()))
print("max promo_days_7d:", int(df["promo_days_7d"].max()), "(<= 7 hona chahiye) | max promo_days_28d:", int(df["promo_days_28d"].max()), "(<= 28)")

print("\nSummary:"); print(df[["promo_streak", "promo_intensity_vs_28d", "store_promo_families", "family_promo_stores"]]
                          .describe().round(2).to_string())

# ek asli promo streak ko aankh se dekho (jahan streak 1 se 5+ tak jaye)
s = df[(df["promo_streak"] >= 5)].iloc[0]
w = df[(df["store_nbr"] == s["store_nbr"]) & (df["family"] == s["family"]) &
       df["date"].between(s["date"] - pd.Timedelta(days=6), s["date"] + pd.Timedelta(days=2))]
print("\nStore", int(s["store_nbr"]), s["family"]); print(w[["date", "onpromotion", "promo_streak", "promo_days_7d"]].to_string(index=False))

shape: (3000888, 53)

NaN per promo feature:
promo_flag                    0
log_onpromotion               0
promo_recorded                0
onpromotion_lag_7d        12474
promo_streak                  0
promo_days_7d                 0
promo_days_28d                0
promo_intensity_vs_28d    12474
store_promo_families          0
family_promo_stores           0

streak == 0 exactly when no promo: True
max promo_days_7d: 7 (<= 7 hona chahiye) | max promo_days_28d: 28 (<= 28)

Summary:
       promo_streak  promo_intensity_vs_28d  store_promo_families  family_promo_stores
count    3000888.00              2988414.00            3000888.00           3000888.00
mean           7.16                    0.26                  6.72                11.00
std           28.14                    1.26                  6.23                17.83
min            0.00                    0.00                  0.00                 0.00
25%            0.00                    0.00                  0.00          

In [5]:
# ---- 1. Transactions: store-level, calendar ke hisaab se HORIZON din purani ----
tx = (df.drop_duplicates(["store_nbr", "date"])
        .pivot(index="date", columns="store_nbr", values="transactions")
        .reindex(full_dates))
tx_lag  = tx.shift(HORIZON)
tx_roll = tx.shift(HORIZON).rolling(7, min_periods=5).mean()

def map_store_date(w):
    s = w.stack(future_stack=True).rename("v").reset_index()
    s.columns = ["date", "store_nbr", "v"]
    return df[["date", "store_nbr"]].merge(s, on=["date", "store_nbr"], how="left")["v"].to_numpy("float32")

df["transactions_lag_7d"]   = map_store_date(tx_lag)
df["transactions_roll_7d"]  = map_store_date(tx_roll)

# ---- 2. Oil: market-level, date ke hisaab se (25 Dec ka gap ffill, sirf past se) ----
oil = (df.drop_duplicates("date").set_index("date")["dcoilwtico"]
         .reindex(full_dates).ffill())
df["dcoilwtico_lag_7d"]      = df["date"].map(oil.shift(HORIZON)).astype("float32")
df["dcoilwtico_roll_28d"]    = df["date"].map(oil.shift(HORIZON).rolling(28, min_periods=14).mean()).astype("float32")

# ---- 3. Store aur family encoding ----
df["store_type_code"] = df["store_type"].astype(str).map({"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}).astype("int8")
df["family_code"]     = df["family"].astype("category").cat.codes.astype("int8")
df["city_code"]       = df["city"].astype("category").cat.codes.astype("int8")
df["state_code"]      = df["state"].astype("category").cat.codes.astype("int8")

PERISHABLE = ["DAIRY", "PRODUCE", "MEATS", "POULTRY", "SEAFOOD", "EGGS", "BREAD/BAKERY", "PREPARED FOODS", "DELI"]
df["is_perishable"] = df["family"].astype(str).isin(PERISHABLE).astype("int8")

# ---- 4. Store open date + reliability flags (EDA ke faislay) ----
st = df.groupby(["store_nbr", "date"])["sales"].sum().reset_index()
open_date = st[st["sales"] > 0].groupby("store_nbr")["date"].min()
df["_open"] = df["store_nbr"].map(open_date)
df["is_store_open"]     = (df["date"] >= df["_open"]).astype("int8")
df["days_since_open"]   = (df["date"] - df["_open"]).dt.days.clip(lower=0).astype("int16")
df["reliable_period"]   = (df["date"] >= "2015-07-15").astype("int8")   # 1 Jun 2015 + 7 din shift + 28 din window
df = df.drop(columns="_open")

# ---- 5. Checks ----
new = ["transactions_lag_7d", "transactions_roll_7d", "dcoilwtico_lag_7d", "dcoilwtico_roll_28d"]
print("shape:", df.shape)
print("\nNaN per new feature:"); print(df[new].isna().sum().to_string())

# transactions lag sahi hai? seedha 7 din pehle ki value se compare
chk = df.sample(5000, random_state=1)[["date", "store_nbr", "transactions_lag_7d"]].copy()
chk["d7"] = chk["date"] - pd.Timedelta(days=7)
ref = df.drop_duplicates(["store_nbr", "date"])[["date", "store_nbr", "transactions"]].rename(columns={"date": "d7"})
ok = chk.merge(ref, on=["d7", "store_nbr"], how="left").dropna()
print("\ntransactions lag matches true value 7 days earlier:",
      bool(np.allclose(ok["transactions_lag_7d"], ok["transactions"], atol=1e-2)), "| rows checked:", len(ok))

print("\nperishable families:", df.loc[df["is_perishable"] == 1, "family"].nunique(), "(9 hona chahiye)")
print("is_store_open share:", round(df["is_store_open"].mean(), 3))
print("reliable_period rows:", int(df["reliable_period"].sum()),
      "| min date:", df.loc[df["reliable_period"] == 1, "date"].min().date())
print("store_type_code values:", sorted(df["store_type_code"].unique()))
print("\nmemory MB:", round(df.memory_usage(deep=True).sum() / 1e6))

shape: (3000888, 65)

NaN per new feature:
transactions_lag_7d     19602
transactions_roll_7d    19602
dcoilwtico_lag_7d       12474
dcoilwtico_roll_28d     35640

transactions lag matches true value 7 days earlier: True | rows checked: 4962

perishable families: 9 (9 hona chahiye)
is_store_open share: 0.926
reliable_period rows: 1356102 | min date: 2015-07-15
store_type_code values: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]

memory MB: 501


In [6]:
# ---- 1. Same-day columns hatao (forecast ke waqt maloom nahi hote) ----
DROP_SAME_DAY = ["transactions", "dcoilwtico", "tx_missing"]
out = df.drop(columns=[c for c in DROP_SAME_DAY if c in df.columns]).copy()
out = out.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)

# ---- 2. Dtype chhote karo (target 'sales' float64 hi rahega) ----
for c in out.select_dtypes("float64").columns:
    if c != "sales":
        out[c] = out[c].astype("float32")

# ---- 3. Leakage check A: rolling_mean_7d haath se dobara nikalo ----
samp = out[out["date"] >= "2015-09-01"].sample(300, random_state=0)
bad = 0
for _, r in samp.iterrows():
    s = wide[(int(r["store_nbr"]), str(r["family"]))]
    w = s.loc[r["date"] - pd.Timedelta(days=13): r["date"] - pd.Timedelta(days=HORIZON)]   # t-13 se t-7 tak
    if not np.isclose(w.mean(), r["rolling_mean_7d"], rtol=1e-3, atol=1e-2, equal_nan=True):
        bad += 1
print("rolling_mean_7d mismatches (sirf date-7 tak ka data use hona chahiye):", bad, "of", len(samp))

# ---- 4. Leakage check B: kisi lag/rolling feature ka naam horizon se chhota na ho ----
import re
lags = [int(re.search(r"lag_(\d+)d", c).group(1)) for c in out.columns if re.search(r"lag_(\d+)d", c)]
print("smallest lag in file:", min(lags), "(>= 7 hona chahiye)")

# ---- 5. Sanity ----
print("\nduplicates (date, store, family):", out.duplicated(["date", "store_nbr", "family"]).sum())
print("shape:", out.shape, "| memory MB:", round(out.memory_usage(deep=True).sum() / 1e6))
print("\nNaN per column (sirf jin mein NaN hai):")
nn = out.isna().sum(); print(nn[nn > 0].to_string())

# ---- 6. Save + reload check ----
from pathlib import Path
OUT = Path("../data/processed"); OUT.mkdir(parents=True, exist_ok=True)
path = OUT / "features_engineered.parquet"
out.to_parquet(path, index=False, compression="zstd")

chk = pd.read_parquet(path)
print("\nreloaded shape:", chk.shape, "| sales sum match:", np.isclose(chk["sales"].sum(), out["sales"].sum()))
print("file size MB:", round(path.stat().st_size / 1e6))

# ---- 7. Column list (data dictionary ke liye) ----
print("\nCOLUMNS:")
print(pd.DataFrame({"dtype": chk.dtypes.astype(str), "nan": chk.isna().sum()}).to_string())

rolling_mean_7d mismatches (sirf date-7 tak ka data use hona chahiye): 0 of 300
smallest lag in file: 7 (>= 7 hona chahiye)

duplicates (date, store, family): 0
shape: (3000888, 62) | memory MB: 474

NaN per column (sirf jin mein NaN hai):
sales_lag_7d              19602
sales_lag_14d             32076
sales_lag_21d             44550
sales_lag_28d             57024
sales_same_dow_mean_4w    12474
rolling_mean_7d           19602
rolling_std_7d            19602
rolling_mean_14d          28512
rolling_std_14d           28512
rolling_mean_28d          48114
rolling_std_28d           48114
onpromotion_lag_7d        12474
promo_intensity_vs_28d    12474
transactions_lag_7d       19602
transactions_roll_7d      19602
dcoilwtico_lag_7d         12474
dcoilwtico_roll_28d       35640

reloaded shape: (3000888, 62) | sales sum match: True
file size MB: 105

COLUMNS:
                                  dtype    nan
id                                int32      0
date                     datetime64[us]

Feature Engineering Summary

The goal is to prepare the cleaned data for predicting supermarket sales for the next 7 days. The model is updated every week. The final dataset has 3,000,888 rows and 62 columns, including 47 new features.

The main rule is that the model should not use information that is not available at prediction time. Sales, transactions and oil prices are taken from at least 7 days earlier. Calendar, holiday and promotion information can be used for the same day because it is known in advance.

1. Calendar and Holiday Features (14)

We added features such as day of the week, month, year, quarter, weekend and payday. We also added special dates like New Year, Christmas, Mother's Day and the 2016 earthquake period because they may affect sales. We included the number of days until the next national holiday and since the previous national holiday.

2. Sales History Features (11)

We used sales from 7, 14, 21 and 28 days earlier. We also calculated their average and the rolling mean and standard deviation for 7, 14 and 28 days. These calculations end 7 days before the prediction date. We use 7 days because weekly orders cannot depend on yesterday's sales. The dates are matched using the calendar so missing dates do not shift the lag values.

3. Promotion Features (10)

We added features to show whether a promotion is active and how many products are on promotion. We also recorded whether promotion data was available how long a promotion continued and how often promotions occurred in the last 7 and 28 days. We measured promotion intensity and the number of product families and stores involved in promotions.

4. Transactions and Oil Features (4)

We used past transaction counts and oil prices including their 7-day lags and rolling averages. We did not use future transaction values because they are not known in advance.

5. Store, Family and Quality Features (8)

We added store type, family, city and state information. We also marked perishable product families, whether a store was open and how many days had passed since it opened. The reliable_period feature marks data from 15 July 2015 onward because a change in data recording was found in the EDA.

Validation

We checked the lag features against actual values from 7 days earlier and they matched. We also manually checked rolling averages on 300 random rows with no errors. The dataset has no duplicate rows and the saved file was reloaded to confirm that sales values remained unchanged.

Output

The final feature dataset is saved as data/processed/features_engineered.parquet. Its size is 105 MB. A data dictionary is also included to explain the features.
